# UD6.01. La capa de conectividad: lo que cuesta mover un dato

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 2 y 3 de los apuntes · Criterios **3.b** y **3.c**

---

Este cuaderno contesta una pregunta que casi siempre se responde de memoria: **por qué el
internet de las cosas no usa HTTP**. La respuesta habitual es «porque HTTP es pesado», y es
verdad, pero no es una respuesta hasta que se dice *cuánto*.

Así que aquí no se cita a nadie. Se construyen los paquetes byte a byte y se cuentan.

Al final del cuaderno tienes que poder contestar estas tres, que son del criterio 3.b y
salen en el cuestionario:

1. ¿Cuántos bytes de sobrecarga añade MQTT a una lectura de temperatura, y cuántos de esos
   los has elegido tú?
2. ¿Qué significa exactamente que el QoS 1 garantiza la entrega *al menos una vez*, y qué
   problema le crea al consumidor?
3. ¿Por qué el modelo publicador/suscriptor permite añadir un consumidor nuevo sin tocar
   los sensores, y por qué eso vale más que ahorrar bytes?

No hace falta instalar nada ni tener un intermediario MQTT: todo es biblioteca estándar.


In [ ]:
from dataclasses import dataclass, field

# Un caso concreto para todo el cuaderno: un almacén frigorífico con sensores de
# temperatura. La lectura es un número con un decimal. Nada más.
TOPICO = "almacen/zona-A/sensor-17/temperatura"
VALOR = b"21.5"

print(f"tópico: {TOPICO!r}  ({len(TOPICO)} caracteres)")
print(f"valor:  {VALOR!r}  ({len(VALOR)} bytes de dato útil)")


---

## 1. Un paquete MQTT, construido a mano

MQTT 3.1.1 es un protocolo binario y su paquete de publicación tiene tres partes:

| Parte | Qué lleva | Cuánto ocupa |
|---|---|---|
| Cabecera fija | Tipo de paquete y banderas, y la longitud de lo que viene detrás | 2 a 5 bytes |
| Cabecera variable | El nombre del tópico, y el identificador de paquete si QoS > 0 | 2 + longitud del tópico (+2) |
| Carga útil | El valor, tal cual, sin envolver en nada | lo que mida el valor |

La longitud se codifica de una forma que vale la pena mirar, porque es donde se ve la
obsesión del protocolo por el tamaño: **siete bits útiles por byte y el octavo como bandera
de continuación**. Así, todo lo que mida menos de 128 bytes gasta un solo byte en decir
cuánto mide.


In [ ]:
def longitud_restante(n):
    '''Codifica un entero como la «longitud restante» de MQTT: siete bits útiles por
    byte, y el octavo bit puesto a 1 si viene otro byte detrás.'''
    salida = bytearray()
    while True:
        byte = n % 128
        n //= 128
        if n > 0:
            byte |= 0x80
        salida.append(byte)
        if n == 0:
            return bytes(salida)


for n in [0, 127, 128, 16_383, 16_384, 2_097_151]:
    codificado = longitud_restante(n)
    print(f"{n:>10,} -> {codificado.hex(' '):<12} ({len(codificado)} bytes)")

print()
print("Con 1 byte se describen mensajes de hasta 127 bytes, que es el caso normal de un")
print("sensor. El máximo del protocolo son 4 bytes: 256 MB en un solo mensaje.")


Ahora el paquete entero. Fíjate en que **no hay ninguna envoltura**: ni JSON, ni cabeceras
de texto, ni nombres de campo. El valor va crudo y el único metadato es el tópico.


In [ ]:
def paquete_publish(topico, carga, qos=0, retener=False, duplicado=False):
    '''Construye un paquete PUBLISH de MQTT 3.1.1 completo.'''
    topico_bytes = topico.encode("utf-8")

    # Cabecera variable: longitud del tópico en 2 bytes, el tópico, y el identificador
    # de paquete solo si hay QoS, porque sin QoS no hay nada que reconocer.
    variable = len(topico_bytes).to_bytes(2, "big") + topico_bytes
    if qos > 0:
        variable += (1).to_bytes(2, "big")

    cuerpo = variable + carga

    # Cabecera fija: los cuatro bits altos son el tipo (3 = PUBLISH) y los cuatro
    # bajos, las banderas.
    banderas = (int(duplicado) << 3) | (qos << 1) | int(retener)
    fija = bytes([(3 << 4) | banderas]) + longitud_restante(len(cuerpo))
    return fija + cuerpo


mqtt0 = paquete_publish(TOPICO, VALOR, qos=0)
print(f"paquete completo ({len(mqtt0)} bytes):")
print(mqtt0.hex(" "))
print()
print("desglose:")
print(f"  cabecera fija        {mqtt0[:2].hex(' '):<20} 2 bytes")
print(f"  longitud del tópico  {mqtt0[2:4].hex(' '):<20} 2 bytes")
print(f"  tópico               {mqtt0[4:4 + len(TOPICO)][:12].hex(' ')}...  "
      f"{len(TOPICO)} bytes")
print(f"  carga útil           {mqtt0[4 + len(TOPICO):].hex(' '):<20} {len(VALOR)} bytes")


Y para estar seguros de que el paquete está bien construido, lo **descodificamos**. Un
codificador que nadie descodifica es un codificador sin comprobar.


In [ ]:
def descodifica_publish(paquete):
    '''Deshace lo que hizo paquete_publish. Devuelve un diccionario con las piezas.'''
    primero = paquete[0]
    assert primero >> 4 == 3, "esto no es un PUBLISH"
    qos = (primero >> 1) & 0b11

    # La longitud restante ocupa un número variable de bytes: se leen mientras el bit
    # alto esté puesto.
    i = 1
    restante = 0
    multiplicador = 1
    while True:
        byte = paquete[i]
        restante += (byte & 127) * multiplicador
        multiplicador *= 128
        i += 1
        if not byte & 0x80:
            break

    n_topico = int.from_bytes(paquete[i:i + 2], "big")
    topico = paquete[i + 2:i + 2 + n_topico].decode("utf-8")
    j = i + 2 + n_topico
    identificador = None
    if qos > 0:
        identificador = int.from_bytes(paquete[j:j + 2], "big")
        j += 2
    return {"qos": qos, "restante": restante, "topico": topico,
            "id": identificador, "carga": paquete[j:]}


descodificado = descodifica_publish(mqtt0)
print(descodificado)

assert descodificado["topico"] == TOPICO
assert descodificado["carga"] == VALOR
assert 1 + 1 + descodificado["restante"] == len(mqtt0)
print()
print("Las tres comprobaciones pasan: el tópico, la carga y la longitud declarada")
print("coinciden con lo que se metió. El paquete es correcto.")


---

## 2. Lo mismo por HTTP, y la cuenta

Ahora la alternativa. Y que quede claro que **no es un hombre de paja**: enviar telemetría
por HTTP a una API REST es lo que se hace cuando no se piensa en el transporte y se
reutiliza la pila web que la empresa ya tiene. Es una decisión razonable hasta que alguien
cuenta los bytes.


In [ ]:
def peticion_http(topico, carga, testigo):
    cuerpo = b'{"valor":' + carga + b'}'
    lineas = [
        f"POST /api/v1/telemetria/{topico} HTTP/1.1",
        "Host: telemetria.ejemplo.es",
        "User-Agent: pasarela-iot/1.4.2",
        "Content-Type: application/json",
        f"Content-Length: {len(cuerpo)}",
        f"Authorization: Bearer {testigo}",
        "Accept: application/json",
        "Connection: keep-alive",
    ]
    return ("\r\n".join(lineas) + "\r\n\r\n").encode("utf-8") + cuerpo


TESTIGO = "e" * 180        # un testigo JWT corto de verdad ronda estos 180 caracteres

mqtt1 = paquete_publish(TOPICO, VALOR, qos=1)
http = peticion_http(TOPICO, VALOR, TESTIGO)

print(http.decode("utf-8")[:330])
print("...")
print()
print(f"{'transporte':<24}{'bytes':>8}{'sobrecarga':>12}{'veces el dato':>15}")
print("-" * 59)
for nombre, paquete in [("MQTT PUBLISH QoS 0", mqtt0),
                        ("MQTT PUBLISH QoS 1", mqtt1),
                        ("HTTP POST + JSON", http)]:
    n = len(paquete)
    print(f"{nombre:<24}{n:>8}{n - len(VALOR):>12}{n / len(VALOR):>15.1f}")


### El matiz que distingue a quien ha medido

De los 40 bytes de sobrecarga de MQTT, **36 son el nombre del tópico**, que lo eliges tú al
diseñar el sistema. La sobrecarga del protocolo en sí son **4 bytes**: dos de cabecera fija
y dos para decir cuánto mide el tópico.

Eso convierte el nombre del tópico en una decisión de diseño con coste, y explica por qué en
instalaciones que pagan por byte —una red móvil de bajo consumo, por ejemplo— los tópicos se
acortan. Vamos a ponerle número a esa decisión.


In [ ]:
protocolo_puro = len(mqtt0) - len(VALOR) - len(TOPICO)
print(f"sobrecarga total de MQTT:      {len(mqtt0) - len(VALOR)} bytes")
print(f"de los cuales, el tópico:      {len(TOPICO)} bytes")
print(f"sobrecarga del protocolo:      {protocolo_puro} bytes")
print()

print("Qué pasa si se acortan los tópicos:")
print(f"{'tópico':<40}{'bytes/mensaje':>15}{'GB/mes*':>10}{'ahorro':>9}")
print("-" * 74)
base = len(paquete_publish(TOPICO, VALOR))
for topico in [TOPICO, "almacen/A/17/temp", "a/A/17/t"]:
    n = len(paquete_publish(topico, VALOR))
    gb_mes = n * 10_000 * 86_400 * 30 / 1e9
    print(f"{topico:<40}{n:>15}{gb_mes:>10.0f}{1 - n / base:>8.0%}")
print()
print("* 10.000 sensores publicando una vez por segundo durante 30 días.")
print()
ahorro = 1 - len(paquete_publish("a/A/17/t", VALOR)) / base
print(f"Ojo con el equilibrio: «a/A/17/t» ahorra un {ahorro:.0%} del tráfico y es")
print("ilegible.")
print("Un tópico ilegible es una decisión que se paga en mantenimiento, no en factura.")
print("La recomendación razonable es la fila del medio.")


---

## 3. La factura, que es el argumento de verdad

Los bytes por mensaje no convencen a nadie. Los euros al mes, sí. Esta es la tabla que hay
que saber construir, porque tiene la forma que pide el criterio 3.a: **una ventaja con un
número y un término de comparación**.


In [ ]:
EUR_POR_GB = 0.09      # tarifa habitual de salida de los grandes proveedores de nube
N_SENSORES = 10_000
HZ = 1.0


def factura(bytes_mensaje, n_sensores=N_SENSORES, hz=HZ, dias=30):
    mensajes = n_sensores * hz * 86_400
    gb_dia = bytes_mensaje * mensajes / 1e9
    return gb_dia, gb_dia * dias, gb_dia * dias * EUR_POR_GB


print(f"{N_SENSORES:,} sensores a {HZ:g} Hz, tarifa {EUR_POR_GB} EUR/GB de salida")
print()
print(f"{'transporte':<24}{'GB/día':>10}{'GB/mes':>10}{'EUR/mes':>12}")
print("-" * 56)
for nombre, paquete in [("MQTT PUBLISH QoS 0", mqtt0),
                        ("MQTT PUBLISH QoS 1", mqtt1),
                        ("HTTP POST + JSON", http)]:
    gb_dia, gb_mes, eur = factura(len(paquete))
    print(f"{nombre:<24}{gb_dia:>10.1f}{gb_mes:>10,.0f}{eur:>12,.0f}")

_, _, eur_mqtt = factura(len(mqtt0))
_, _, eur_http = factura(len(http))
print()
print(f"Diferencia: {eur_http / eur_mqtt:.1f} veces, {eur_http - eur_mqtt:,.0f} EUR al mes,")
print(f"{(eur_http - eur_mqtt) * 12:,.0f} EUR al año, por una decisión que se toma")
print("en una tarde y que después nadie vuelve a revisar.")


### Lo que esta cuenta NO incluye, y hay que decirlo

Un número sin sus límites declarados no vale. Este cálculo deja fuera tres cosas, y las tres
van **en contra de HTTP**, así que la conclusión es conservadora:

1. **El establecimiento de la conexión.** Un apretón de manos de TLS ronda los 5 kB. MQTT lo
   paga una vez y mantiene la conexión abierta; HTTP sin conexión persistente lo paga en
   cada mensaje.
2. **Las respuestas.** HTTP contesta a cada petición con otra respuesta con sus cabeceras.
   MQTT con QoS 0 no contesta nada.
3. **El coste de proceso en el servidor**, que crece con el número de peticiones, no con los
   bytes.

Vamos a ponerle número al primero, que es el mayor con diferencia.


In [ ]:
TLS_APRETON = 5_000      # bytes, aproximado, de un apretón de manos TLS 1.3 completo

print("Escenario A: HTTP sin conexión persistente (un apretón por mensaje)")
gb_dia, gb_mes, eur_a = factura(len(http) + TLS_APRETON)
print(f"  {gb_mes:>12,.0f} GB/mes   {eur_a:>10,.0f} EUR/mes")

print()
print("Escenario B: MQTT, un apretón por dispositivo y por día")
bytes_dia = len(mqtt0) * N_SENSORES * HZ * 86_400 + TLS_APRETON * N_SENSORES
gb_mes_b = bytes_dia * 30 / 1e9
eur_b = gb_mes_b * EUR_POR_GB
print(f"  {gb_mes_b:>12,.0f} GB/mes   {eur_b:>10,.0f} EUR/mes")

print()
print(f"Con el apretón contado, la diferencia sube a {eur_a / eur_b:.0f} veces.")
print("El apretón de TLS le cuesta a MQTT una fracción inapreciable, y a HTTP sin")
print("conexión persistente, más que todos los datos juntos. Por eso mantener la")
print("conexión abierta es LA decisión de diseño de MQTT, no el formato binario.")


---

## 4. Publicador/suscriptor: la ventaja que no es de tamaño

Hasta aquí todo ha sido contar bytes, y contar bytes es la parte fácil. La ventaja de diseño
de MQTT es otra y vale más:

> **El que publica no sabe quién le lee, y el que lee no sabe quién publica.**

Con HTTP, el sensor tiene que conocer la dirección del servidor. Si mañana hace falta que
los datos lleguen también al equipo de mantenimiento, hay que **tocar los sensores**, y
tocar diez mil sensores instalados en una nave no es una tarea de programación: es una obra.

Con publicador/suscriptor, el consumidor nuevo se suscribe y ya está.

Vamos a construir el intermediario, que son cuarenta líneas, para ver que el mecanismo no
tiene nada dentro. Lo único que tiene miga es el emparejamiento de comodines.


In [ ]:
def empareja(filtro, topico):
    '''Reglas de MQTT para los comodines de suscripción:
       «+» sustituye exactamente un nivel;
       «#» sustituye el resto de niveles y solo puede ir al final.'''
    f = filtro.split("/")
    t = topico.split("/")
    for i, parte in enumerate(f):
        if parte == "#":
            return i == len(f) - 1
        if i >= len(t):
            return False
        if parte != "+" and parte != t[i]:
            return False
    return len(f) == len(t)


casos = [
    ("almacen/zona-A/sensor-17/temperatura", "almacen/zona-A/sensor-17/temperatura", True),
    ("almacen/+/sensor-17/temperatura", "almacen/zona-A/sensor-17/temperatura", True),
    ("almacen/+/temperatura", "almacen/zona-A/sensor-17/temperatura", False),
    ("almacen/#", "almacen/zona-A/sensor-17/temperatura", True),
    ("almacen/zona-A/#", "almacen/zona-B/sensor-1/temperatura", False),
    ("#", "cualquier/cosa/de/la/instalacion", True),
    ("almacen", "almacen/zona-A", False),
]
print(f"{'filtro':<38}{'tópico':<42}{'esperado':>9}{'sale':>7}")
print("-" * 96)
for filtro, topico, esperado in casos:
    sale = empareja(filtro, topico)
    print(f"{filtro:<38}{topico:<42}{str(esperado):>9}{str(sale):>7}")
    assert sale == esperado, (filtro, topico)

print()
print("La tercera fila es la que más se falla: «+» es UN nivel, no «lo que sea».")
print("Y la última, «#» suscrito a todo, es el comodín que hay que PROHIBIR en la")
print("configuración del intermediario. Se vuelve sobre esto en los apuntes, bloque 8.")


In [ ]:
@dataclass
class Intermediario:
    '''Un intermediario MQTT de juguete: solo reparte. No hay red, ni sesiones, ni
    persistencia. Es lo único que hace falta para ver el desacoplamiento.'''
    suscripciones: list = field(default_factory=list)
    entregados: int = 0

    def suscribe(self, filtro, nombre, funcion):
        self.suscripciones.append((filtro, nombre, funcion))
        print(f"  [{nombre}] suscrito a {filtro!r}")

    def publica(self, paquete):
        d = descodifica_publish(paquete)
        for filtro, nombre, funcion in self.suscripciones:
            if empareja(filtro, d["topico"]):
                funcion(d["topico"], d["carga"])
                self.entregados += 1


intermediario = Intermediario()

print("Al principio solo está el cuadro de mando:")
intermediario.suscribe("almacen/+/+/temperatura", "cuadro-de-mando",
                       lambda t, c: print(f"      cuadro: {t.split('/')[1]} = {c.decode()}"))

print("\nEl sensor publica, y no sabe quién le escucha:")
intermediario.publica(paquete_publish(TOPICO, b"21.5"))

print("\nAhora entra mantenimiento. NO se toca el sensor:")
intermediario.suscribe("almacen/#", "mantenimiento",
                       lambda t, c: print(f"      mantenimiento: registra {t}"))

print("\nEl sensor publica exactamente igual que antes:")
intermediario.publica(paquete_publish(TOPICO, b"21.8"))

print(f"\nEntregas totales: {intermediario.entregados}. El código del sensor no ha")
print("cambiado ni una línea. ESA es la ventaja, y es de arquitectura, no de bytes.")


---

## 5. Calidad de servicio: qué se puede perder

MQTT ofrece tres garantías de entrega, y elegir una es una decisión de negocio disfrazada de
decisión técnica.

| QoS | Garantía | Mensajes de red | Cuándo |
|---|---|---|---|
| 0 | Como mucho una vez | 1 | Telemetría frecuente donde perder una lectura no importa |
| 1 | **Al menos** una vez | 2 | El caso por defecto |
| 2 | Exactamente una vez | 4 | Órdenes que no se pueden repetir |

La trampa está en el 1. *Al menos una vez* significa **que puede llegar dos veces**: si el
reconocimiento se pierde, el emisor reenvía. Y si el consumidor suma lo que recibe, el
reenvío cuenta doble.


In [ ]:
def consumidor_ingenuo(estado, lectura):
    '''Suma todo lo que le llega. Con QoS 1 esto está mal y no da ningún error.'''
    estado["total"] += lectura["valor"]
    estado["n"] += 1


def consumidor_idempotente(estado, lectura):
    '''Descarta lo que ya ha visto. Procesar dos veces el mismo mensaje da el mismo
    resultado que procesarlo una: eso es ser idempotente.'''
    if lectura["id"] in estado["vistos"]:
        estado["descartados"] += 1
        return
    estado["vistos"].add(lectura["id"])
    estado["total"] += lectura["valor"]
    estado["n"] += 1


# Diez lecturas, de las que tres se reenvían porque se perdió el reconocimiento.
lecturas = [{"id": i, "valor": 20.0 + i * 0.1} for i in range(10)]
recibidas = lecturas + [lecturas[2], lecturas[5], lecturas[7]]

ingenuo = {"total": 0.0, "n": 0}
idempotente = {"total": 0.0, "n": 0, "vistos": set(), "descartados": 0}
for lectura in recibidas:
    consumidor_ingenuo(ingenuo, lectura)
    consumidor_idempotente(idempotente, lectura)

verdad = sum(l["valor"] for l in lecturas) / len(lecturas)
print(f"media real                     {verdad:.4f}")
print(f"consumidor ingenuo             {ingenuo['total'] / ingenuo['n']:.4f}   "
      f"({ingenuo['n']} lecturas contadas)")
print(f"consumidor idempotente         "
      f"{idempotente['total'] / idempotente['n']:.4f}   "
      f"({idempotente['n']} contadas, {idempotente['descartados']} descartadas)")

assert abs(idempotente["total"] / idempotente["n"] - verdad) < 1e-9
print()
print("El error del ingenuo es pequeño, y por eso es peligroso: no da error, no rompe")
print("nada, y la media queda mal para siempre. Es el mismo tipo de fallo silencioso")
print("que la UD5 buscaba en el modelo roto.")
print()
print("Y fíjate en la solución: NO es subir a QoS 2, que cuadruplica el tráfico. Es")
print("poner un identificador por lectura y comprobarlo al recibir. La garantía se")
print("consigue en el consumidor, que es donde sale barata.")


---

## 6. La tabla de la unidad

Cierra el cuaderno con la comparación completa. Es la tabla que tendrás que reutilizar en la
actividad A6.1, así que compruébala.


In [ ]:
filas = [
    ("MQTT", "publicador/suscriptor", len(mqtt0), "TCP",
     "telemetría de muchos dispositivos"),
    ("HTTP", "petición/respuesta", len(http), "TCP",
     "APIs, pocos mensajes, red buena"),
    ("CoAP", "petición/respuesta", 4 + 2 + len(TOPICO) + len(VALOR), "UDP",
     "dispositivos con muy poca memoria"),
    ("WebSocket", "bidireccional", 6 + len(VALOR), "TCP",
     "del servidor al navegador"),
]
print(f"{'protocolo':<12}{'modelo':<24}{'bytes*':>8}{'sobre':>7}  {'dónde encaja'}")
print("-" * 95)
for nombre, modelo, n, transporte, donde in filas:
    print(f"{nombre:<12}{modelo:<24}{n:>8}{transporte:>7}  {donde}")
print()
print("* para la misma lectura de 4 bytes con el mismo tópico. La fila de CoAP es una")
print("  estimación de su cabecera mínima de 4 bytes más la opción de ruta; la de")
print("  WebSocket, de una trama pequeña con máscara. Las dos primeras están medidas")
print("  byte a byte en este cuaderno; las dos últimas, no: no las informes como si")
print("  lo estuvieran.")


---

## Lo que hay que llevarse

1. **MQTT añade 4 bytes de protocolo.** Los otros 36 de sobrecarga son el nombre del
   tópico, y lo eliges tú.
2. **HTTP cuesta diez veces más en tráfico** para el mismo dato, y con el apretón de TLS
   contado, mucho más.
3. **La ventaja grande de MQTT no es el tamaño, es el desacoplamiento**: un consumidor nuevo
   no obliga a tocar los sensores.
4. **QoS 1 entrega *al menos* una vez**, y la defensa correcta no es QoS 2, es un consumidor
   idempotente.
5. **`+` es un nivel y `#` es el resto.** Una suscripción a `#` ve la instalación entera, y
   por eso es lo primero que hay que prohibir en el intermediario.

### Para la actividad A6.1

Necesitas la tabla de la sección 3 rehecha **con los números de tu caso**: tu número de
dispositivos, tu frecuencia y tu tamaño de mensaje. La tarifa por gigabyte búscala en la
lista de precios pública de un proveedor y **cita de dónde la has sacado y de qué fecha es**.
